> # ⚠️ ARCHIVED — DO NOT CITE ANY NUMBER FROM THIS NOTEBOOK
>
> This notebook is from the project's first generation (2026-08-11). It is kept
> to show the methodological path, **not** as evidence.
>
> - Its stored outputs have been **stripped**, deliberately, so no figure or table
>   here can be mistaken for a current result. Git history retains them.
> - It reads data paths and episode identifiers that **no longer exist**, so it
>   cannot be re-executed to regenerate them.
> - Where it uses the Grand Ouest reference, note that the reference has since been
>   re-resolved onto a new episode reconstruction, and the linkage methods,
>   thresholds, and splits all changed afterwards. Same source data, different
>   everything else.
>
> Current evidence lives in `notebooks/10`–`14`, the `*.md` reports at the
> repository root, and `reports/boamp_methodology_chapter.pdf`.


# BOAMP Data Preprocessing & Data Engineering

## tl;dr

This notebook prepares an analysis-ready, contract-level engineering layer from the canonical raw BOAMP CSV. It is the preprocessing stage after raw acquisition/EDA and before renewal-linkage, manual evaluation, survival analysis, NLP modeling, or change-point detection.

The notebook intentionally keeps the raw BOAMP file unchanged. It creates derived datasets in `data/processed/boamp/` and validation metadata in `data/metadata/boamp_preprocessing_summary.json`.

## Context & Methods

### Scope

This notebook performs data engineering only:

- read the canonical raw BOAMP CSV in chunks;
- validate raw schema and historical date coverage;
- parse date columns into reproducible engineered date features;
- normalize text fields for matching/modeling;
- parse list-like raw CSV cells such as departments, descriptors, and market types;
- extract useful nested fields from raw `donnees`, especially CPV codes, SIREN/SIRET candidates, date candidates, and amount candidates;
- build transparent digital-contract signals from CPV prefixes and keyword hits;
- write compact processed Parquet datasets and a manual-annotation sample.

### What This Notebook Does Not Do

It does **not** link renewals, infer ground truth, train NLP models, run survival models, remove duplicates, or decide final statistical validity. Those are separate downstream notebooks.

### Key Assumptions

- Source data: `data/raw/boamp/boamp_2015_2025_raw.csv`.
- Historical interval: `2015-01-01 <= dateparution < 2026-01-01`.
- Digital first-pass CPV families follow the internship guide: `32`, `35`, `48`, and `72`.
- Keyword rules are only candidate-generation signals. They are not final labels.

In [ ]:
from __future__ import annotations

import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

## Data

### 1. Configure Paths and Parameters

The default run processes the full raw CSV. Set `MAX_ROWS = 10000` during development if you need a quick debug pass.

In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/raw/boamp/boamp_2015_2025_raw.csv").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
RAW_CSV_PATH = PROJECT_ROOT / "data/raw/boamp/boamp_2015_2025_raw.csv"
FIELDS_PATH = PROJECT_ROOT / "data/metadata/boamp_fields.json"
RAW_CSV_SUMMARY_PATH = PROJECT_ROOT / "data/metadata/boamp_raw_csv_summary.json"
DOWNLOAD_SUMMARY_PATH = PROJECT_ROOT / "data/metadata/boamp_download_summary.json"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/boamp"
METADATA_DIR = PROJECT_ROOT / "data/metadata"

CORE_PARQUET_PATH = PROCESSED_DIR / "boamp_preprocessed_core_2015_2025.parquet"
DIGITAL_PARQUET_PATH = PROCESSED_DIR / "boamp_digital_candidates_2015_2025.parquet"
ANNOTATION_SAMPLE_PATH = PROCESSED_DIR / "boamp_digital_annotation_sample_500.csv"
PREPROCESSING_SUMMARY_PATH = METADATA_DIR / "boamp_preprocessing_summary.json"

START_YEAR = 2015
END_YEAR = 2025
HISTORICAL_START = "2015-01-01"
HISTORICAL_END_EXCLUSIVE = "2026-01-01"
CHUNKSIZE = 50_000
MAX_ROWS = None
ANNOTATION_SAMPLE_SIZE = 500
RANDOM_SEED = 42

for directory in [PROCESSED_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

assert RAW_CSV_PATH.exists(), RAW_CSV_PATH
assert FIELDS_PATH.exists(), FIELDS_PATH
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw CSV: {RAW_CSV_PATH}")
print(f"Processed output: {PROCESSED_DIR}")

### 2. Load Raw Metadata and Show a Small Preview

The preview reads only 20 rows. The full preprocessing below uses chunked reads.

In [ ]:
field_metadata = json.loads(FIELDS_PATH.read_text(encoding="utf-8"))
raw_field_names = [field["name"] for field in field_metadata]
raw_csv_summary = json.loads(RAW_CSV_SUMMARY_PATH.read_text(encoding="utf-8")) if RAW_CSV_SUMMARY_PATH.exists() else {}
download_summary = json.loads(DOWNLOAD_SUMMARY_PATH.read_text(encoding="utf-8")) if DOWNLOAD_SUMMARY_PATH.exists() else {}
expected_rows = download_summary.get("global_validation", {}).get("historical_downloaded_count") or raw_csv_summary.get("rows")

raw_preview_df = pd.read_csv(RAW_CSV_PATH, nrows=20, dtype="string", encoding="utf-8", low_memory=False)
print(f"Raw preview shape: {raw_preview_df.shape}")
print(f"Raw field count from metadata: {len(raw_field_names)}")
display(raw_preview_df.head(20))

## Engineering Rules

### 3. Define Taxonomy, Regexes, and Helper Functions

The digital-contract signal is intentionally broad. It is a candidate-generation feature, not a final supervised label.

In [ ]:
RAW_COLUMNS_NEEDED = [
    "idweb",
    "id",
    "contractfolderid",
    "objet",
    "filename",
    "famille",
    "famille_libelle",
    "dateparution",
    "datefindiffusion",
    "datelimitereponse",
    "nomacheteur",
    "titulaire",
    "type_procedure",
    "soustype_procedure",
    "procedure_libelle",
    "nature",
    "sousnature",
    "nature_libelle",
    "descripteur_code",
    "descripteur_libelle",
    "dc",
    "type_marche",
    "type_avis",
    "code_departement",
    "code_departement_prestation",
    "source_schema",
    "donnees",
    "url_avis",
]
RAW_COLUMNS_NEEDED = [column for column in RAW_COLUMNS_NEEDED if column in raw_field_names]

DIGITAL_CPV_PREFIX2 = {"32", "35", "48", "72"}
DIGITAL_KEYWORDS = {
    "cloud": ["cloud", "nuage", "iaas", "paas", "saas", "hebergement", "hébergement", "datacenter", "data center", "centre de donnees", "centre de données"],
    "cybersecurity": ["cyber", "cybersecurite", "cybersécurité", "securite informatique", "sécurité informatique", "pare-feu", "firewall", "antivirus", "centre operationnel de securite", "centre opérationnel de sécurité", "systeme de securite informatique", "système de sécurité informatique"],
    "software": ["logiciel", "logiciels", "software", "progiciel", "applicatif", "licence logiciel", "licences logicielles", "erp", "crm", "saas"],
    "it_services": ["informatique", "systeme d'information", "système d'information", "infogerance", "infogérance", "maintenance informatique", "assistance utilisateurs", "support informatique"],
    "telecom_network": ["telecommunication", "télécommunication", "telecom", "télécom", "fibre optique", "wifi", "wi-fi", "wan", "vpn", "connectivite", "connectivité", "reseau informatique", "réseau informatique", "reseaux informatiques", "réseaux informatiques"],
    "ai_data": ["intelligence artificielle", "machine learning", "apprentissage automatique", "big data", "science des donnees", "science des données", "entrepot de donnees", "entrepôt de données", "gouvernance des donnees", "gouvernance des données"],
}
DATE_RE = re.compile(r"\d{4}-\d{2}-\d{2}")
CPV_RE = re.compile(r"^\d{8}$")
SIREN_RE = re.compile(r"(?<!\d)(\d{9})(?!\d)")
SIRET_RE = re.compile(r"(?<!\d)(\d{14})(?!\d)")
NUMBER_RE = re.compile(r"[-+]?\d[\d\s.,]*")


def strip_accents(value: str) -> str:
    normalized = unicodedata.normalize("NFKD", value)
    return "".join(char for char in normalized if not unicodedata.combining(char))


def normalize_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)) or pd.isna(value):
        return ""
    text = strip_accents(str(value).lower())
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_buyer(value: Any) -> str:
    return normalize_text(value).upper()


NORMALIZED_DIGITAL_KEYWORDS = sorted({normalize_text(keyword): keyword for values in DIGITAL_KEYWORDS.values() for keyword in values}.items(), key=lambda item: len(item[0]), reverse=True)


def parse_json_cell(value: Any) -> Any:
    if value is None or (isinstance(value, float) and math.isnan(value)) or pd.isna(value):
        return None
    text = str(value).strip()
    if text == "":
        return None
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


def parse_list_cell(value: Any) -> list[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)) or pd.isna(value):
        return []
    text = str(value).strip()
    if text == "":
        return []
    if text.startswith("["):
        try:
            parsed = json.loads(text)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed if item not in [None, ""]]
        except json.JSONDecodeError:
            pass
    return [text]


def walk_json(obj: Any):
    if isinstance(obj, dict):
        for key, value in obj.items():
            yield str(key), value
            yield from walk_json(value)
    elif isinstance(obj, list):
        for item in obj:
            yield from walk_json(item)


def collect_cpv_codes_from_obj(obj: Any) -> list[str]:
    codes = set()

    def collect_under_cpv(value: Any):
        if isinstance(value, dict):
            for nested_value in value.values():
                collect_under_cpv(nested_value)
        elif isinstance(value, list):
            for item in value:
                collect_under_cpv(item)
        elif isinstance(value, (str, int, float)):
            candidate = str(value).strip()
            if CPV_RE.fullmatch(candidate):
                codes.add(candidate)

    if obj is None:
        return []
    for key, value in walk_json(obj):
        if "CPV" in key.upper():
            collect_under_cpv(value)
    return sorted(codes)


def collect_identifier_candidates(obj: Any) -> tuple[list[str], list[str]]:
    sirens = set()
    sirets = set()
    if obj is None:
        return [], []
    for key, value in walk_json(obj):
        upper_key = key.upper()
        if "SIREN" in upper_key or "SIRET" in upper_key:
            text = str(value)
            sirens.update(SIREN_RE.findall(text))
            sirets.update(SIRET_RE.findall(text))
    return sorted(sirens), sorted(sirets)


def collect_date_candidates(obj: Any) -> tuple[list[str], list[str]]:
    start_dates = set()
    end_dates = set()
    if obj is None:
        return [], []
    for key, value in walk_json(obj):
        upper_key = key.upper()
        if "DATE_DEBUT" in upper_key or "DEBUT" == upper_key:
            start_dates.update(DATE_RE.findall(str(value)))
        if "DATE_FIN" in upper_key or "FIN" == upper_key:
            end_dates.update(DATE_RE.findall(str(value)))
    return sorted(start_dates), sorted(end_dates)


def parse_amount_number(text: str) -> float | None:
    match = NUMBER_RE.search(text)
    if not match:
        return None
    number = match.group(0).replace(" ", "").replace(",", ".")
    try:
        return float(number)
    except ValueError:
        return None


def collect_amount_candidates(obj: Any, limit: int = 8) -> list[float]:
    amounts = []
    if obj is None:
        return []
    for key, value in walk_json(obj):
        upper_key = key.upper()
        if any(token in upper_key for token in ["MONTANT", "VALEUR", "ESTIMATION"]):
            if isinstance(value, (str, int, float)):
                parsed = parse_amount_number(str(value))
                if parsed is not None and parsed >= 0:
                    amounts.append(parsed)
            elif isinstance(value, dict):
                for nested_value in value.values():
                    if isinstance(nested_value, (str, int, float)):
                        parsed = parse_amount_number(str(nested_value))
                        if parsed is not None and parsed >= 0:
                            amounts.append(parsed)
        if len(amounts) >= limit:
            break
    # keep candidates, do not decide a canonical amount here
    return amounts[:limit]


def keyword_hits(normalized_text: str) -> list[str]:
    padded = f" {normalized_text} "
    hits = []
    for normalized_keyword, original_keyword in NORMALIZED_DIGITAL_KEYWORDS:
        if normalized_keyword and f" {normalized_keyword} " in padded:
            hits.append(original_keyword)
    return sorted(set(hits))


def compact_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, separators=(",", ":"))

### 4. Transform One Chunk

This function turns raw BOAMP rows into engineered rows. Heavy raw text fields are not carried into the processed core table, but every row remains traceable through `idweb` and `url_avis`.

In [ ]:
def transform_chunk(raw_chunk: pd.DataFrame) -> pd.DataFrame:
    engineered = pd.DataFrame(index=raw_chunk.index)

    for column in [
        "idweb", "id", "contractfolderid", "objet", "filename", "famille", "famille_libelle",
        "nomacheteur", "titulaire", "type_procedure", "soustype_procedure", "procedure_libelle",
        "nature", "sousnature", "nature_libelle", "code_departement_prestation", "source_schema", "url_avis",
    ]:
        engineered[column] = raw_chunk[column] if column in raw_chunk else pd.NA

    for date_column in ["dateparution", "datefindiffusion", "datelimitereponse"]:
        engineered[date_column] = raw_chunk[date_column] if date_column in raw_chunk else pd.NA
        parsed = pd.to_datetime(engineered[date_column], errors="coerce", utc=True if date_column == "datelimitereponse" else False)
        if date_column == "datelimitereponse":
            engineered[f"{date_column}_parsed"] = parsed.dt.strftime("%Y-%m-%dT%H:%M:%SZ")
        else:
            engineered[f"{date_column}_parsed"] = parsed.dt.strftime("%Y-%m-%d")

    publication_date = pd.to_datetime(engineered["dateparution"], errors="coerce")
    response_deadline = pd.to_datetime(engineered["datelimitereponse"], errors="coerce", utc=True)
    publication_date_utc = pd.to_datetime(engineered["dateparution"], errors="coerce", utc=True)
    engineered["publication_year"] = publication_date.dt.year.astype("Int64")
    engineered["publication_month"] = publication_date.dt.to_period("M").astype("string")
    engineered["publication_quarter"] = publication_date.dt.to_period("Q").astype("string")
    engineered["days_to_response_deadline"] = ((response_deadline - publication_date_utc).dt.total_seconds() / 86400).round(2)

    engineered["buyer_name_raw"] = engineered["nomacheteur"]
    engineered["buyer_name_normalized"] = engineered["nomacheteur"].map(normalize_buyer)
    engineered["objet_normalized"] = engineered["objet"].map(normalize_text)

    engineered["code_departement_list_json"] = raw_chunk.get("code_departement", pd.Series(pd.NA, index=raw_chunk.index)).map(lambda value: compact_json(parse_list_cell(value)))
    engineered["descripteur_code_list_json"] = raw_chunk.get("descripteur_code", pd.Series(pd.NA, index=raw_chunk.index)).map(lambda value: compact_json(parse_list_cell(value)))
    engineered["descripteur_libelle_list_json"] = raw_chunk.get("descripteur_libelle", pd.Series(pd.NA, index=raw_chunk.index)).map(lambda value: compact_json(parse_list_cell(value)))
    engineered["dc_list_json"] = raw_chunk.get("dc", pd.Series(pd.NA, index=raw_chunk.index)).map(lambda value: compact_json(parse_list_cell(value)))
    engineered["type_marche_list_json"] = raw_chunk.get("type_marche", pd.Series(pd.NA, index=raw_chunk.index)).map(lambda value: compact_json(parse_list_cell(value)))
    engineered["type_avis_list_json"] = raw_chunk.get("type_avis", pd.Series(pd.NA, index=raw_chunk.index)).map(lambda value: compact_json(parse_list_cell(value)))

    parsed_donnees = raw_chunk.get("donnees", pd.Series(pd.NA, index=raw_chunk.index)).map(parse_json_cell)
    cpv_codes = parsed_donnees.map(collect_cpv_codes_from_obj)
    identifiers = parsed_donnees.map(collect_identifier_candidates)
    date_candidates = parsed_donnees.map(collect_date_candidates)
    amount_candidates = parsed_donnees.map(collect_amount_candidates)

    engineered["cpv_codes_json"] = cpv_codes.map(compact_json)
    engineered["cpv_prefix2_json"] = cpv_codes.map(lambda values: compact_json(sorted({value[:2] for value in values if len(value) >= 2})))
    engineered["primary_cpv"] = cpv_codes.map(lambda values: values[0] if values else pd.NA)
    engineered["primary_cpv_prefix2"] = cpv_codes.map(lambda values: values[0][:2] if values else pd.NA)
    engineered["siren_candidates_json"] = identifiers.map(lambda pair: compact_json(pair[0]))
    engineered["siret_candidates_json"] = identifiers.map(lambda pair: compact_json(pair[1]))
    engineered["primary_siren_candidate"] = identifiers.map(lambda pair: pair[0][0] if pair[0] else pd.NA)
    engineered["primary_siret_candidate"] = identifiers.map(lambda pair: pair[1][0] if pair[1] else pd.NA)
    engineered["contract_start_date_candidates_json"] = date_candidates.map(lambda pair: compact_json(pair[0]))
    engineered["contract_end_date_candidates_json"] = date_candidates.map(lambda pair: compact_json(pair[1]))
    engineered["primary_contract_start_date"] = date_candidates.map(lambda pair: pair[0][0] if pair[0] else pd.NA)
    engineered["primary_contract_end_date"] = date_candidates.map(lambda pair: pair[1][0] if pair[1] else pd.NA)
    engineered["amount_candidates_json"] = amount_candidates.map(compact_json)
    engineered["amount_candidate_count"] = amount_candidates.map(len).astype("int64")

    contract_start = pd.to_datetime(engineered["primary_contract_start_date"], errors="coerce")
    contract_end = pd.to_datetime(engineered["primary_contract_end_date"], errors="coerce")
    engineered["declared_duration_days"] = (contract_end - contract_start).dt.days.astype("Int64")

    combined_text = (
        engineered["objet"].fillna("").astype(str) + " "
        + engineered["nomacheteur"].fillna("").astype(str) + " "
        + raw_chunk.get("descripteur_libelle", pd.Series("", index=raw_chunk.index)).fillna("").astype(str) + " "
        + engineered["procedure_libelle"].fillna("").astype(str) + " "
        + engineered["nature_libelle"].fillna("").astype(str)
    ).map(normalize_text)
    hits = combined_text.map(keyword_hits)
    cpv_prefix_sets = cpv_codes.map(lambda values: {value[:2] for value in values if len(value) >= 2})
    engineered["digital_keyword_hits_json"] = hits.map(compact_json)
    engineered["digital_keyword_hit_count"] = hits.map(len).astype("int64")
    engineered["is_digital_by_keyword"] = hits.map(lambda values: len(values) > 0)
    engineered["is_digital_by_cpv"] = cpv_prefix_sets.map(lambda prefixes: bool(prefixes & DIGITAL_CPV_PREFIX2))
    engineered["is_digital_candidate"] = engineered["is_digital_by_keyword"] | engineered["is_digital_by_cpv"]
    engineered["digital_signal_count"] = engineered[["is_digital_by_keyword", "is_digital_by_cpv"]].astype(int).sum(axis=1)

    nullable_int_columns = {"publication_year", "declared_duration_days"}
    float_columns = {"days_to_response_deadline"}
    integer_columns = {"amount_candidate_count", "digital_keyword_hit_count", "digital_signal_count"}
    boolean_columns = {"is_digital_by_keyword", "is_digital_by_cpv", "is_digital_candidate"}

    for column in engineered.columns:
        if column in nullable_int_columns:
            engineered[column] = engineered[column].astype("Int64")
        elif column in float_columns:
            engineered[column] = pd.to_numeric(engineered[column], errors="coerce").astype("float64")
        elif column in integer_columns:
            engineered[column] = pd.to_numeric(engineered[column], errors="coerce").fillna(0).astype("int64")
        elif column in boolean_columns:
            engineered[column] = engineered[column].fillna(False).astype("bool")
        else:
            engineered[column] = engineered[column].astype("string")

    return engineered.reset_index(drop=True)

## Processing

### 5. Run the Chunked Preprocessing Pipeline

The core table contains all historical BOAMP rows with engineered features. The digital-candidate table is a subset for later annotation, NLP classification, trend analysis, and renewal-linkage experiments.

In [ ]:
def remove_if_exists(path: Path):
    if path.exists():
        path.unlink()

remove_if_exists(CORE_PARQUET_PATH)
remove_if_exists(DIGITAL_PARQUET_PATH)
remove_if_exists(ANNOTATION_SAMPLE_PATH)
remove_if_exists(PREPROCESSING_SUMMARY_PATH)

core_writer = None
digital_writer = None
rng = np.random.default_rng(RANDOM_SEED)
annotation_sample_frames = []

processed_rows = 0
digital_candidate_rows = 0
missing_idweb = 0
duplicate_idweb = 0
seen_idweb = set()
outside_historical_range = 0
invalid_dateparution = 0
min_dateparution = None
max_dateparution = None
yearly_counts = Counter()
digital_yearly_counts = Counter()
cpv_prefix_counter = Counter()
keyword_counter = Counter()
source_schema_counter = Counter()
missing_core_counter = Counter()
started_at = datetime.now()

reader = pd.read_csv(
    RAW_CSV_PATH,
    usecols=RAW_COLUMNS_NEEDED,
    chunksize=CHUNKSIZE,
    nrows=MAX_ROWS,
    dtype="string",
    encoding="utf-8",
    low_memory=False,
)

for chunk_index, raw_chunk in enumerate(reader, start=1):
    engineered = transform_chunk(raw_chunk)
    processed_rows += len(engineered)

    idweb_series = engineered["idweb"].fillna("").astype(str)
    missing_idweb += int((idweb_series == "").sum())
    for value in idweb_series[idweb_series != ""]:
        if value in seen_idweb:
            duplicate_idweb += 1
        else:
            seen_idweb.add(value)

    date_series = pd.to_datetime(engineered["dateparution"], errors="coerce")
    invalid_dateparution += int(date_series.isna().sum())
    valid_dates = date_series.dropna()
    if len(valid_dates):
        chunk_min = valid_dates.min().strftime("%Y-%m-%d")
        chunk_max = valid_dates.max().strftime("%Y-%m-%d")
        min_dateparution = chunk_min if min_dateparution is None else min(min_dateparution, chunk_min)
        max_dateparution = chunk_max if max_dateparution is None else max(max_dateparution, chunk_max)
        outside_historical_range += int(((valid_dates < pd.Timestamp(HISTORICAL_START)) | (valid_dates >= pd.Timestamp(HISTORICAL_END_EXCLUSIVE))).sum())
        yearly_counts.update(valid_dates.dt.year.astype(str))

    digital_frame = engineered[engineered["is_digital_candidate"]].copy()
    digital_candidate_rows += len(digital_frame)
    if len(digital_frame):
        digital_yearly_counts.update(digital_frame["publication_year"].dropna().astype(str))

    for value in engineered["cpv_prefix2_json"].dropna():
        cpv_prefix_counter.update(json.loads(value))
    for value in engineered["digital_keyword_hits_json"].dropna():
        keyword_counter.update(json.loads(value))
    source_schema_counter.update(engineered["source_schema"].dropna().astype(str))
    missing_core_counter.update((engineered.isna() | engineered.eq("")).sum().astype(int).to_dict())

    # Deterministic-ish reservoir-style sample from digital candidates for future manual annotation.
    if len(digital_frame):
        sample_fraction = min(1.0, ANNOTATION_SAMPLE_SIZE / max(digital_candidate_rows, ANNOTATION_SAMPLE_SIZE))
        annotation_sample_frames.append(digital_frame.sample(frac=sample_fraction, random_state=RANDOM_SEED + chunk_index))

    core_table = pa.Table.from_pandas(engineered, preserve_index=False)
    if core_writer is None:
        core_writer = pq.ParquetWriter(CORE_PARQUET_PATH, core_table.schema, compression="zstd")
    core_writer.write_table(core_table)

    if len(digital_frame):
        digital_table = pa.Table.from_pandas(digital_frame, preserve_index=False)
        if digital_writer is None:
            digital_writer = pq.ParquetWriter(DIGITAL_PARQUET_PATH, digital_table.schema, compression="zstd")
        digital_writer.write_table(digital_table)

    if chunk_index % 5 == 0:
        elapsed = (datetime.now() - started_at).total_seconds() / 60
        print(f"Processed {processed_rows:,} rows in {elapsed:.1f} minutes; digital candidates so far: {digital_candidate_rows:,}")

if core_writer is not None:
    core_writer.close()
if digital_writer is not None:
    digital_writer.close()

if annotation_sample_frames:
    annotation_pool = pd.concat(annotation_sample_frames, ignore_index=True)
    annotation_sample = (
        annotation_pool
        .drop_duplicates("idweb")
        .sample(n=min(ANNOTATION_SAMPLE_SIZE, annotation_pool["idweb"].nunique()), random_state=RANDOM_SEED)
        [[
            "idweb", "dateparution", "buyer_name_raw", "objet", "famille_libelle", "procedure_libelle",
            "nature_libelle", "primary_cpv", "cpv_codes_json", "digital_keyword_hits_json",
            "is_digital_by_cpv", "is_digital_by_keyword", "url_avis",
        ]]
        .sort_values(["dateparution", "idweb"])
    )
    annotation_sample.insert(0, "human_label_digital_contract", "")
    annotation_sample.insert(1, "human_label_segment", "")
    annotation_sample.insert(2, "human_notes", "")
    annotation_sample.to_csv(ANNOTATION_SAMPLE_PATH, index=False, encoding="utf-8")

elapsed_minutes = (datetime.now() - started_at).total_seconds() / 60
print(f"Finished preprocessing {processed_rows:,} rows in {elapsed_minutes:.1f} minutes")
print(f"Digital candidates: {digital_candidate_rows:,}")

## Checks

### 6. Validate Outputs and Save Summary

These checks ensure the preprocessing pass covered the same historical raw corpus and did not introduce accidental date leakage.

In [ ]:
core_parquet = pq.ParquetFile(CORE_PARQUET_PATH)
core_output_rows = core_parquet.metadata.num_rows
if DIGITAL_PARQUET_PATH.exists():
    digital_output_rows = pq.ParquetFile(DIGITAL_PARQUET_PATH).metadata.num_rows
else:
    digital_output_rows = 0

validation_checks = {
    "processed_rows": int(processed_rows),
    "expected_rows": int(expected_rows) if expected_rows is not None else None,
    "processed_rows_match_expected": bool(expected_rows is None or processed_rows == expected_rows),
    "core_parquet_rows": int(core_output_rows),
    "core_rows_match_processed": bool(core_output_rows == processed_rows),
    "digital_candidate_rows": int(digital_candidate_rows),
    "digital_parquet_rows": int(digital_output_rows),
    "digital_rows_match_processed": bool(digital_output_rows == digital_candidate_rows),
    "unique_idweb": int(len(seen_idweb)),
    "duplicate_idweb": int(duplicate_idweb),
    "missing_idweb": int(missing_idweb),
    "min_dateparution": min_dateparution,
    "max_dateparution": max_dateparution,
    "outside_historical_range": int(outside_historical_range),
    "invalid_dateparution": int(invalid_dateparution),
}

summary_payload = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_raw_csv": str(RAW_CSV_PATH),
    "source_rows_expected": expected_rows,
    "parameters": {
        "chunksize": CHUNKSIZE,
        "max_rows": MAX_ROWS,
        "historical_start": HISTORICAL_START,
        "historical_end_exclusive": HISTORICAL_END_EXCLUSIVE,
        "digital_cpv_prefix2": sorted(DIGITAL_CPV_PREFIX2),
        "digital_keyword_taxonomy": DIGITAL_KEYWORDS,
    },
    "outputs": {
        "core_parquet": str(CORE_PARQUET_PATH),
        "digital_candidates_parquet": str(DIGITAL_PARQUET_PATH),
        "annotation_sample_csv": str(ANNOTATION_SAMPLE_PATH),
    },
    "validation": validation_checks,
    "yearly_counts": dict(sorted(yearly_counts.items())),
    "digital_yearly_counts": dict(sorted(digital_yearly_counts.items())),
    "top_cpv_prefix2": dict(cpv_prefix_counter.most_common(30)),
    "top_digital_keywords": dict(keyword_counter.most_common(50)),
    "top_source_schemas": dict(source_schema_counter.most_common(20)),
    "missing_engineered_fields": dict(sorted(missing_core_counter.items())),
}
PREPROCESSING_SUMMARY_PATH.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

pd.DataFrame([
    ["Processed rows", f"{validation_checks['processed_rows']:,}"],
    ["Expected rows", f"{validation_checks['expected_rows']:,}" if validation_checks['expected_rows'] is not None else None],
    ["Rows match expected", validation_checks["processed_rows_match_expected"]],
    ["Core parquet rows", f"{validation_checks['core_parquet_rows']:,}"],
    ["Digital candidate rows", f"{validation_checks['digital_candidate_rows']:,}"],
    ["Unique idweb", f"{validation_checks['unique_idweb']:,}"],
    ["Duplicate idweb", validation_checks["duplicate_idweb"]],
    ["Missing idweb", validation_checks["missing_idweb"]],
    ["Date coverage", f"{validation_checks['min_dateparution']} to {validation_checks['max_dateparution']}"],
    ["Outside historical range", validation_checks["outside_historical_range"]],
    ["Invalid dateparution", validation_checks["invalid_dateparution"]],
], columns=["check", "value"])

### 7. Inspect Processed Samples

The first table is the engineered core layer. The second table is the candidate digital-contract subset that will feed annotation and later NLP/linkage work.

In [ ]:
core_sample = pd.read_parquet(CORE_PARQUET_PATH, columns=[
    "idweb", "dateparution", "publication_year", "buyer_name_raw", "objet", "primary_cpv",
    "cpv_prefix2_json", "digital_keyword_hits_json", "is_digital_by_cpv", "is_digital_by_keyword", "is_digital_candidate", "url_avis",
]).head(20)
display(core_sample)

if DIGITAL_PARQUET_PATH.exists():
    digital_sample = pd.read_parquet(DIGITAL_PARQUET_PATH, columns=[
        "idweb", "dateparution", "buyer_name_raw", "objet", "primary_cpv", "cpv_codes_json",
        "digital_keyword_hits_json", "is_digital_by_cpv", "is_digital_by_keyword", "url_avis",
    ]).head(20)
    display(digital_sample)

### 8. Review High-Level Engineering Distributions

These lightweight tables are sanity checks for the engineered digital candidate layer.

In [ ]:
yearly_table = pd.DataFrame(sorted(yearly_counts.items()), columns=["year", "raw_records"])
digital_yearly_table = pd.DataFrame(sorted(digital_yearly_counts.items()), columns=["year", "digital_candidate_records"])
coverage_table = yearly_table.merge(digital_yearly_table, on="year", how="left").fillna({"digital_candidate_records": 0})
coverage_table["digital_candidate_share"] = coverage_table["digital_candidate_records"] / coverage_table["raw_records"]

top_cpv_prefix_table = pd.DataFrame(cpv_prefix_counter.most_common(20), columns=["cpv_prefix2", "records_or_occurrences"])
top_keyword_table = pd.DataFrame(keyword_counter.most_common(20), columns=["keyword", "records_or_occurrences"])

display(coverage_table)
display(top_cpv_prefix_table)
display(top_keyword_table)

## Next Steps

The outputs from this notebook are ready for the next project stages:

- manual digital-contract annotation using `boamp_digital_annotation_sample_500.csv`;
- refinement of the digital taxonomy;
- candidate generation for renewal linkage;
- linkage algorithm comparison and manual evaluation;
- survival-analysis dataset construction with observed/censored durations.